# PP-OCRv6 DET → Group line → VietOCR wordlist gating V2 → Vintern line/group fallback

Pipeline cho 1 frame ảnh:

```text
PP-OCRv6_medium_det
→ line boxes
→ group line-box gần/chồng nhau
→ crop line + crop group context
→ VietOCR recognition
→ return_prob nếu hỗ trợ; nếu missing/flat thì dùng rec-missing gating
→ wordlist lexical features: lex_ratio + diacritic_suspicious
→ composite_score từ rec_conf/det_score/lex_ratio/diacritic_susp/charset_penalty
→ structural filter: timestamp/logo/60 giây/bottom-counter
→ ranked Vintern line fallback
→ ranked Vintern group fallback cho group nhiều dòng/context khó
→ group_text_clean chỉ chứa text đã accept
→ group_text_review giữ text chưa chắc đúng, không dùng để index chính
→ export CSV/JSON/TXT/visualization
```

V2 sửa các lỗi từ run gần nhất:

- Sửa lỗi `dynamic_preprocess` khiến Vintern không chạy thật.
- Thêm group-level Vintern OCR cho crop nhiều dòng.
- Sửa trường hợp `VietOCR return_prob` không trả confidence (`return_prob_missing`).
- Nếu `rec_conf` missing/flat: chỉ auto-accept dòng dài có `det_score + lex_ratio` tốt; dòng ngắn/tên riêng/biển báo vẫn đi VLM.
- Không fallback `group_text` sang review text khi `num_keep_lines=0`.


In [ ]:
# CELL 1 — Optional install + environment notes
# Nếu môi trường đã OK, giữ RUN_INSTALL=False.
# Nếu Vintern lỗi do transformers quá mới, set RUN_INSTALL=True, chạy cell này, Restart Runtime, rồi chạy lại từ CELL 2.

RUN_INSTALL = False

if RUN_INSTALL:
    !pip uninstall -y pillow PIL paddleocr paddlex paddlepaddle paddlepaddle-gpu vietocr transformers accelerate tokenizers -q
    !pip install -q --no-cache-dir "pillow==10.4.0"
    !pip install -q --no-cache-dir paddlepaddle paddleocr vietocr
    !pip install -q --no-cache-dir "transformers==4.45.2" "accelerate==0.34.2" "tokenizers==0.20.3"
    !pip install -q --no-cache-dir sentencepiece timm einops torchvision opencv-python-headless pandas matplotlib tqdm numpy

print('Install cell done. Nếu RUN_INSTALL=True, hãy Restart Runtime trước khi chạy tiếp.')


In [ ]:
# CELL 2 — Environment flags, imports
import os
os.environ['FLAGS_use_mkldnn'] = '0'
os.environ['FLAGS_enable_mkldnn'] = '0'
os.environ['FLAGS_allocator_strategy'] = 'auto_growth'
# Nếu không tải được Paddle model từ HuggingFace, bật dòng dưới:
# os.environ['PADDLE_PDX_MODEL_SOURCE'] = 'BOS'

import time, json, re, unicodedata
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageEnhance
from tqdm.auto import tqdm
from IPython.display import display

import PIL, paddle, torch, transformers
from paddleocr import TextDetection
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg

print('Pillow:', PIL.__version__)
print('Paddle:', paddle.__version__, '| Paddle CUDA:', paddle.is_compiled_with_cuda())
print('Torch:', torch.__version__, '| Torch CUDA:', torch.cuda.is_available())
print('Transformers:', transformers.__version__)


In [ ]:
# CELL 3 — Input frame
IMAGE_PATH = '/content/frame.jpg'  # TODO: đổi thành ảnh của bạn
assert Path(IMAGE_PATH).exists(), f'Không tìm thấy ảnh: {IMAGE_PATH}'

img_bgr = cv2.imread(IMAGE_PATH)
assert img_bgr is not None, f'Không đọc được ảnh: {IMAGE_PATH}'
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W = img_rgb.shape[:2]

plt.figure(figsize=(14, 8))
plt.imshow(img_rgb)
plt.axis('off')
plt.title(f'{IMAGE_PATH} | shape={img_rgb.shape}')
plt.show()


In [ ]:
# CELL 4 — Config
CONFIG = {
    'det_model_name': 'PP-OCRv6_medium_det',
    'det_limit_type': 'min',
    'det_limit_side_len': 960,
    'det_thresh': 0.30,
    'det_box_thresh': 0.50,
    'det_unclip_ratio': 1.8,

    'drop_low_det_score_below': 0.0,
    'min_box_width': 10,
    'min_box_height': 8,
    'min_box_aspect_ratio': 0.25,

    'perspective_padding': 12,
    'crop_min_height_for_ocr': 72,
    'crop_upscale_max_factor': 3.5,
    'add_white_border': 10,
    'contrast_factor': 1.25,

    'group_min_x_overlap_ratio': 0.12,
    'group_max_vertical_gap_ratio': 1.80,
    'group_min_lines': 2,
    'group_crop_padding': 28,

    'vietocr_config': 'vgg_transformer',
    'vietocr_beamsearch': True,
    'use_vietocr_return_prob': True,
    'rec_conf_fallback_when_missing': 0.50,
    'rec_conf_flat_std_threshold': 1e-4,
    'rec_conf_flat_unique_ratio_threshold': 0.20,

    'wordlist_paths': [
        '/content/drive/MyDrive/vietnamese/vn_dictionary.txt',
        '/content/drive/MyDrive/vietnamese/general_dict.txt',
        '/content/vn_dictionary.txt',
        '/content/general_dict.txt',
        '/mnt/data/vn_dictionary.txt',
        '/mnt/data/general_dict.txt',
    ],
    'wordlist_min_token_len': 2,
    'wordlist_min_entries_to_enable': 1000,
    'neutral_lex_ratio_if_no_wordlist': 0.50,

    'feature_weights': {
        'rec_conf': 0.42,
        'det_score': 0.18,
        'lex_ratio': 0.22,
        'diacritic_susp': -0.12,
        'charset_penalty': -0.06,
    },
    'composite_bias': 0.00,

    'weak_det_score_threshold': 0.60,
    'auto_accept_rec_conf': 0.90,
    'auto_accept_det_score': 0.70,
    'auto_accept_lex_ratio': 0.50,
    'auto_accept_max_diacritic_susp': 0.00,
    'auto_accept_max_charset_penalty': 0.05,

    'escalate_composite_threshold': 0.68,
    'escalate_rec_conf_threshold': 0.85,
    'escalate_lex_ratio_threshold': 0.45,
    'escalate_diacritic_susp_threshold': 0.30,

    # Rule riêng khi VietOCR confidence missing/flat.
    'rec_missing_auto_accept_min_tokens': 5,
    'rec_missing_auto_accept_det_score': 0.78,
    'rec_missing_auto_accept_lex_ratio': 0.70,
    'rec_missing_auto_accept_max_diacritic_susp': 0.05,
    'rec_missing_auto_accept_max_charset_penalty': 0.02,

    'min_quality_for_clean_text': 68,

    'use_vintern_fallback': True,
    'vintern_model_name': '5CD-AI/Vintern-1B-v3_5',
    'vintern_max_candidates': 8,
    'vintern_max_tiles': 3,
    'vintern_image_size': 448,
    'vintern_max_new_tokens': 128,

    'vintern_agreement_similarity': 0.82,
    'vintern_min_composite_accept': 0.68,
    'vintern_score_margin_to_override': 0.06,
    'vintern_keep_review_on_disagreement': True,

    'use_vintern_group_fallback': True,
    'vintern_group_max_candidates': 4,
    'vintern_group_min_lines': 2,
    'vintern_group_max_new_tokens': 160,
    'group_vlm_min_vlm_lines': 1,
    'group_vlm_need_review_only': True,
    'group_vintern_min_composite_accept': 0.55,
    'group_vintern_accept_even_if_disagree': True,

    'content_value_weight': 0.10,
    'content_value_max_bonus': 0.20,

    'allow_group_text_fallback_to_review': False,
    'filter_60giay_variants': True,

    'max_visualize_line_crops': 24,
    'max_visualize_groups': 16,
}
CONFIG


In [ ]:
# CELL 5 — Init PP-OCRv6_medium_det + VietOCR
PADDLE_DEVICE = 'gpu:0' if paddle.is_compiled_with_cuda() else 'cpu'
print('Paddle detector device:', PADDLE_DEVICE)

detector = TextDetection(
    model_name=CONFIG['det_model_name'],
    device=PADDLE_DEVICE,
    engine='paddle_static',
    limit_side_len=CONFIG['det_limit_side_len'],
    limit_type=CONFIG['det_limit_type'],
    thresh=CONFIG['det_thresh'],
    box_thresh=CONFIG['det_box_thresh'],
    unclip_ratio=CONFIG['det_unclip_ratio'],
    enable_mkldnn=False,
    cpu_threads=4,
)

viet_cfg = Cfg.load_config_from_name(CONFIG['vietocr_config'])
viet_cfg['device'] = 'cuda:0' if torch.cuda.is_available() else 'cpu'
viet_cfg['predictor']['beamsearch'] = CONFIG['vietocr_beamsearch']
vietocr_predictor = Predictor(viet_cfg)

print('PP-OCRv6 detector initialized.')
print('VietOCR initialized:', viet_cfg['device'])


In [ ]:
# CELL 6 — Init Vintern fallback
USE_VINTERN = CONFIG.get('use_vintern_fallback', True)
vintern_model = None
vintern_tokenizer = None
VINTERN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability()
    VINTERN_DTYPE = torch.bfloat16 if major >= 8 else torch.float16
else:
    VINTERN_DTYPE = torch.float32

print('Vintern enabled:', USE_VINTERN)
print('Vintern device:', VINTERN_DEVICE)
print('Vintern dtype:', VINTERN_DTYPE)

if USE_VINTERN:
    try:
        import torchvision.transforms as T
        from torchvision.transforms.functional import InterpolationMode
        from transformers import AutoModel, AutoTokenizer, PreTrainedModel
        if not hasattr(PreTrainedModel, 'all_tied_weights_keys'):
            PreTrainedModel.all_tied_weights_keys = {}

        vintern_tokenizer = AutoTokenizer.from_pretrained(
            CONFIG['vintern_model_name'], trust_remote_code=True, use_fast=False
        )
        vintern_model = AutoModel.from_pretrained(
            CONFIG['vintern_model_name'],
            torch_dtype=VINTERN_DTYPE,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
            use_flash_attn=False,
        ).eval().to(VINTERN_DEVICE)
        print('Loaded Vintern:', CONFIG['vintern_model_name'])
    except Exception as e:
        print('WARNING: Không load được Vintern. Pipeline vẫn chạy bằng VietOCR.')
        print('Vintern error:', repr(e))
        print('Gợi ý: RUN_INSTALL=True ở CELL 1 để pin transformers==4.45.2, sau đó Restart Runtime.')
        USE_VINTERN = False


In [ ]:
# CELL 6.1 — Vintern dynamic_preprocess fix
# Required by vintern_ocr_crop() and vintern_ocr_group_crop().
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def build_transform(input_size):
    return transforms.Compose([
        transforms.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
        transforms.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, image_size):
    best_ratio_diff = float('inf')
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect_ratio = ratio[0] / ratio[1]
        ratio_diff = abs(aspect_ratio - target_aspect_ratio)
        if ratio_diff < best_ratio_diff:
            best_ratio_diff = ratio_diff
            best_ratio = ratio
        elif ratio_diff == best_ratio_diff:
            if area > 0.5 * image_size * image_size * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio


def dynamic_preprocess(image, min_num=1, max_num=6, image_size=448, use_thumbnail=True):
    orig_width, orig_height = image.size
    aspect_ratio = orig_width / max(orig_height, 1)
    target_ratios = set()
    for n in range(min_num, max_num + 1):
        for i in range(1, n + 1):
            for j in range(1, n + 1):
                if min_num <= i * j <= max_num:
                    target_ratios.add((i, j))
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    target_aspect_ratio = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_width, orig_height, image_size)
    target_width = image_size * target_aspect_ratio[0]
    target_height = image_size * target_aspect_ratio[1]
    blocks = target_aspect_ratio[0] * target_aspect_ratio[1]
    resized_img = image.resize((target_width, target_height), Image.Resampling.BICUBIC)
    processed_images = []
    for i in range(blocks):
        box = (
            (i % target_aspect_ratio[0]) * image_size,
            (i // target_aspect_ratio[0]) * image_size,
            ((i % target_aspect_ratio[0]) + 1) * image_size,
            ((i // target_aspect_ratio[0]) + 1) * image_size,
        )
        processed_images.append(resized_img.crop(box))
    if use_thumbnail and len(processed_images) != 1:
        processed_images.append(image.resize((image_size, image_size), Image.Resampling.BICUBIC))
    transform = build_transform(image_size)
    return torch.stack([transform(img) for img in processed_images])

print('dynamic_preprocess is ready.')


In [ ]:
# CELL 7 — Geometry, OCR and grouping utilities

def to_jsonable(obj):
    if isinstance(obj, np.ndarray): return obj.tolist()
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, dict): return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, list): return [to_jsonable(v) for v in obj]
    return obj

def normalize_text(text):
    text = '' if text is None else str(text)
    text = unicodedata.normalize('NFC', text).strip()
    return re.sub(r'\s+', ' ', text)

def safe_get_first(inner, keys, default=None):
    for key in keys:
        if isinstance(inner, dict) and key in inner and inner[key] is not None:
            return inner[key]
    return default

def parse_text_detection_output(det_output):
    boxes, scores = [], []
    for res in det_output:
        data = getattr(res, 'json', None)
        if callable(data): data = data()
        if data is None and hasattr(res, 'to_dict'): data = res.to_dict()
        if not isinstance(data, dict): data = getattr(res, 'res', None)
        if isinstance(data, dict):
            inner = data.get('res', data)
            polys = safe_get_first(inner, ['dt_polys', 'rec_polys', 'boxes'], default=[])
            det_scores = safe_get_first(inner, ['dt_scores', 'scores'], default=[])
            if isinstance(polys, np.ndarray): polys = polys.tolist()
            if isinstance(det_scores, np.ndarray): det_scores = det_scores.tolist()
            for i, poly in enumerate(polys):
                score = float(det_scores[i]) if i < len(det_scores) else None
                boxes.append(np.array(poly, dtype=np.float32))
                scores.append(score)
    return boxes, scores

def poly_to_xyxy(poly):
    poly = np.array(poly, dtype=np.float32)
    xs, ys = poly[:, 0], poly[:, 1]
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]

def box_hw(poly):
    x1, y1, x2, y2 = poly_to_xyxy(poly)
    return max(1.0, x2-x1), max(1.0, y2-y1)

def x_overlap_ratio(b1, b2):
    x11, y11, x12, y12 = b1
    x21, y21, x22, y22 = b2
    inter = max(0.0, min(x12, x22) - max(x11, x21))
    return inter / min(max(1.0, x12-x11), max(1.0, x22-x21))

def vertical_gap(b1, b2):
    x11, y11, x12, y12 = b1
    x21, y21, x22, y22 = b2
    if y12 < y21: return y21-y12
    if y22 < y11: return y11-y22
    return 0.0

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.parent[rb] = ra

def should_group_stacked_boxes(a, b, cfg):
    b1, b2 = a['bbox_xyxy'], b['bbox_xyxy']
    x_ov = x_overlap_ratio(b1, b2)
    y_gap = vertical_gap(b1, b2)
    h1, h2 = max(1.0, b1[3]-b1[1]), max(1.0, b2[3]-b2[1])
    return x_ov >= cfg['group_min_x_overlap_ratio'] and y_gap <= ((h1+h2)/2.0) * cfg['group_max_vertical_gap_ratio']

def build_stacked_groups(line_items, cfg):
    uf = UnionFind(len(line_items))
    for i in range(len(line_items)):
        for j in range(i+1, len(line_items)):
            if should_group_stacked_boxes(line_items[i], line_items[j], cfg): uf.union(i, j)
    group_map = {}
    for i in range(len(line_items)):
        group_map.setdefault(uf.find(i), []).append(i)
    groups = []
    for gid, indices in enumerate(group_map.values()):
        lines = [line_items[i] for i in indices]
        x1 = min(x['bbox_xyxy'][0] for x in lines); y1 = min(x['bbox_xyxy'][1] for x in lines)
        x2 = max(x['bbox_xyxy'][2] for x in lines); y2 = max(x['bbox_xyxy'][3] for x in lines)
        scores = [x['det_score'] for x in lines if x.get('det_score') is not None]
        groups.append({
            'group_id': gid, 'line_indices': indices, 'num_lines': len(indices),
            'is_multiline_group': len(indices) >= cfg['group_min_lines'],
            'bbox_xyxy': [float(x1), float(y1), float(x2), float(y2)],
            'mean_det_score': float(np.mean(scores)) if scores else None,
        })
    groups = sorted(groups, key=lambda g: (g['bbox_xyxy'][1], g['bbox_xyxy'][0]))
    for order, g in enumerate(groups): g['reading_order'] = order
    return groups

def assign_group_info_to_lines(line_items, groups):
    for g in groups:
        for idx in g['line_indices']:
            line_items[idx]['group_id'] = g['group_id']
            line_items[idx]['group_num_lines'] = g['num_lines']
            line_items[idx]['is_multiline_group'] = g['is_multiline_group']
            line_items[idx]['group_bbox_xyxy'] = g['bbox_xyxy']
    return line_items

def crop_axis_from_bbox(image_rgb, bbox, pad):
    h, w = image_rgb.shape[:2]
    x1, y1, x2, y2 = bbox
    x1 = max(0, int(x1)-pad); y1 = max(0, int(y1)-pad)
    x2 = min(w-1, int(x2)+pad); y2 = min(h-1, int(y2)+pad)
    if x2 <= x1 or y2 <= y1: return None
    return image_rgb[y1:y2, x1:x2]

def order_points(pts):
    pts = np.array(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)
    s = pts.sum(axis=1); diff = np.diff(pts, axis=1).reshape(-1)
    rect[0] = pts[np.argmin(s)]; rect[2] = pts[np.argmax(s)]
    rect[1] = pts[np.argmin(diff)]; rect[3] = pts[np.argmax(diff)]
    return rect

def crop_perspective_line(image_rgb, box, pad):
    h, w = image_rgb.shape[:2]
    rect = order_points(box)
    cx, cy = rect[:,0].mean(), rect[:,1].mean()
    expanded = rect.copy()
    for i in range(4):
        vec = expanded[i] - np.array([cx, cy], dtype=np.float32)
        expanded[i] += vec / (np.linalg.norm(vec)+1e-6) * pad
    expanded[:,0] = np.clip(expanded[:,0], 0, w-1)
    expanded[:,1] = np.clip(expanded[:,1], 0, h-1)
    max_w = max(1, int(max(np.linalg.norm(expanded[2]-expanded[3]), np.linalg.norm(expanded[1]-expanded[0]))))
    max_h = max(1, int(max(np.linalg.norm(expanded[1]-expanded[2]), np.linalg.norm(expanded[0]-expanded[3]))))
    dst = np.array([[0,0],[max_w-1,0],[max_w-1,max_h-1],[0,max_h-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(expanded.astype(np.float32), dst)
    return cv2.warpPerspective(image_rgb, M, (max_w, max_h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)

def prepare_crop_for_ocr(crop_rgb, min_height, max_upscale_factor, border, contrast_factor):
    if crop_rgb is None or crop_rgb.size == 0: return None
    pil = Image.fromarray(crop_rgb).convert('RGB')
    w, h = pil.size
    if h < min_height:
        scale = min(max_upscale_factor, min_height/max(h,1))
        pil = pil.resize((max(1,int(w*scale)), max(1,int(h*scale))), Image.Resampling.BICUBIC)
    pil = ImageEnhance.Contrast(pil).enhance(contrast_factor)
    if border > 0: pil = ImageOps.expand(pil, border=border, fill=(255,255,255))
    return pil

def sort_lines_in_group_by_rows(group, line_items, y_threshold_ratio=0.55):
    lines = [line_items[i] for i in group['line_indices']]
    rows = []
    if not lines: return rows
    heights = [max(1.0, x['bbox_xyxy'][3]-x['bbox_xyxy'][1]) for x in lines]
    y_thresh = max(8.0, float(np.median(heights))*y_threshold_ratio)
    enriched = []
    for line in lines:
        x1,y1,x2,y2 = line['bbox_xyxy']
        enriched.append((line, (x1+x2)/2.0, (y1+y2)/2.0))
    enriched.sort(key=lambda x: x[2])
    for line, cx, cy in enriched:
        placed = False
        for row in rows:
            if abs(row['cy']-cy) <= y_thresh:
                row['items'].append((line,cx,cy)); row['cy'] = float(np.mean([v[2] for v in row['items']]))
                placed = True; break
        if not placed: rows.append({'cy': cy, 'items': [(line,cx,cy)]})
    rows.sort(key=lambda r: r['cy'])
    sorted_rows = []
    for row in rows:
        row['items'].sort(key=lambda x: x[1])
        sorted_rows.append([x[0] for x in row['items']])
    return sorted_rows

def build_group_text_with_newlines(group, line_items, text_key='final_text', include_filtered=False):
    row_texts = []
    for row in sort_lines_in_group_by_rows(group, line_items):
        texts = []
        for line in row:
            if not include_filtered and line.get('is_filtered', False): continue
            txt = normalize_text(line.get(text_key, ''))
            if txt: texts.append(txt)
        if texts: row_texts.append(' '.join(texts))
    return '\n'.join(row_texts)


In [ ]:
# CELL 8.1 — Wordlist, composite scoring, filters, Vintern helpers

import difflib

VN_DIACRITIC_CHARS = set(
    'àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩ'
    'òóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ'
    'ÀÁẠẢÃÂẦẤẬẨẪĂẰẮẶẲẴÈÉẸẺẼÊỀẾỆỂỄÌÍỊỈĨ'
    'ÒÓỌỎÕÔỒỐỘỔỖƠỜỚỢỞỠÙÚỤỦŨƯỪỨỰỬỮỲÝỴỶỸĐ'
)

ALLOWED_CHARS = (
    set('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789')
    | set('.,:;!?-/()[]%&+ ')
    | VN_DIACRITIC_CHARS
)

TOKEN_RE = re.compile(r"[0-9A-Za-zÀ-ỹĐđ]+(?:[-'][0-9A-Za-zÀ-ỹĐđ]+)*")


def strip_diacritics(text):
    text = '' if text is None else str(text)
    text = text.replace('Đ', 'D').replace('đ', 'd')
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return unicodedata.normalize('NFC', text)


def normalize_token(token):
    token = normalize_text(token).lower()
    token = token.strip(".,:;!?/()[]{}'\"“”‘’`~|\\")
    return token


def extract_tokens(text):
    text = normalize_text(text)
    return [normalize_token(m.group(0)) for m in TOKEN_RE.finditer(text) if normalize_token(m.group(0))]


def load_vietnamese_wordlist(paths):
    wordset = set()
    base_to_variants = {}

    for p in paths:
        path = Path(p)
        if not path.exists():
            continue

        try:
            lines = path.read_text(encoding='utf-8', errors='ignore').splitlines()
        except Exception as e:
            print(f'WARNING: cannot read wordlist {path}: {e}')
            continue

        for line in lines:
            for token in extract_tokens(line):
                if len(token) < CONFIG.get('wordlist_min_token_len', 2):
                    continue
                if sum(ch.isalpha() for ch in token) == 0:
                    continue

                wordset.add(token)
                base = strip_diacritics(token)
                base_to_variants.setdefault(base, set()).add(token)

        print(f'Loaded wordlist: {path} | current entries={len(wordset)}')

    return wordset, base_to_variants


VN_WORDSET, VN_BASE_TO_VARIANTS = load_vietnamese_wordlist(CONFIG.get('wordlist_paths', []))
WORDLIST_ENABLED = len(VN_WORDSET) >= CONFIG.get('wordlist_min_entries_to_enable', 1000)

print('WORDLIST_ENABLED:', WORDLIST_ENABLED)
print('VN_WORDSET size:', len(VN_WORDSET))
print('VN_BASE_TO_VARIANTS size:', len(VN_BASE_TO_VARIANTS))


def postprocess_ocr_text(text):
    text = normalize_text(text)
    text = re.sub(r'^(\d{1,2}):(\d{2})[.](\d{2})$', r'\1:\2:\3', text)
    text = re.sub(r'\bklh\s*[:;]?\s*oly\b', 'KLH: OLY', text, flags=re.I)
    text = re.sub(r'\bklt\s*[:;]?\s*1?copolyod\b', 'KLT: 1COPOLYOD', text, flags=re.I)
    text = re.sub(r'\bklv\s*[:;]?\s*1?copolyod\b', 'KLV: 1COPOLYOD', text, flags=re.I)
    text = re.sub(r'\bKLVICOPOLYOD\b', 'KLV: 1COPOLYOD', text, flags=re.I)
    if re.fullmatch(r'c[:：]o', text, flags=re.I):
        text = 'C:0'
    return text


def is_timestamp_text(text):
    return re.fullmatch(r'\d{1,2}[:.]\d{2}([:.]\d{2})?', normalize_text(text)) is not None


def is_top_right_logo_region(bbox, H, W):
    if not isinstance(bbox, list) or len(bbox) != 4:
        return False
    x1, y1, x2, y2 = bbox
    return x1 > W * 0.72 and y2 < H * 0.22


def is_bottom_counter(text, bbox, H, W):
    text = normalize_text(text).lower()
    if not isinstance(bbox, list) or len(bbox) != 4:
        return False
    x1, y1, x2, y2 = bbox
    if y1 <= H * 0.72:
        return False
    return text in {'60', '69', '6o', 'giây', 'giay', 'giấy'} or (len(text) <= 3 and any(ch.isdigit() for ch in text))


def normalize_conf_value(conf):
    if conf is None:
        return None

    if isinstance(conf, (list, tuple, np.ndarray)):
        vals = []
        for x in conf:
            try:
                vals.append(float(x))
            except Exception:
                pass
        if not vals:
            return None
        return float(np.mean(vals))

    try:
        return float(conf)
    except Exception:
        return None


def lexical_features(text):
    tokens = extract_tokens(text)
    eval_tokens = [
        t for t in tokens
        if len(t) >= CONFIG.get('wordlist_min_token_len', 2)
        and not t.isdigit()
    ]

    if not eval_tokens:
        return {
            'tokens': tokens,
            'eval_tokens': eval_tokens,
            'lex_ratio': 1.0 if tokens else 0.0,
            'diacritic_susp': 0.0,
            'lex_hits': 0,
            'num_eval_tokens': 0,
            'diacritic_suspicious_tokens': [],
            'oov_tokens': [],
        }

    if not WORDLIST_ENABLED:
        return {
            'tokens': tokens,
            'eval_tokens': eval_tokens,
            'lex_ratio': CONFIG.get('neutral_lex_ratio_if_no_wordlist', 0.50),
            'diacritic_susp': 0.0,
            'lex_hits': 0,
            'num_eval_tokens': len(eval_tokens),
            'diacritic_suspicious_tokens': [],
            'oov_tokens': eval_tokens,
        }

    lex_hits = 0
    suspicious = []
    oov = []

    for t in eval_tokens:
        base = strip_diacritics(t)

        if t in VN_WORDSET:
            lex_hits += 1
            continue

        # Nếu base tồn tại nhưng token không thuộc biến thể có dấu hợp lệ => nghi lỗi dấu
        if base in VN_BASE_TO_VARIANTS:
            suspicious.append(t)
        else:
            oov.append(t)

    lex_ratio = lex_hits / max(1, len(eval_tokens))
    diacritic_susp = len(suspicious) / max(1, len(eval_tokens))

    return {
        'tokens': tokens,
        'eval_tokens': eval_tokens,
        'lex_ratio': float(lex_ratio),
        'diacritic_susp': float(diacritic_susp),
        'lex_hits': int(lex_hits),
        'num_eval_tokens': int(len(eval_tokens)),
        'diacritic_suspicious_tokens': suspicious,
        'oov_tokens': oov,
    }


def charset_penalty(text):
    text = normalize_text(text)
    if not text:
        return 1.0
    bad = sum(ch not in ALLOWED_CHARS for ch in text)
    return float(bad / max(1, len(text)))


def repeated_garbage_penalty(text):
    text = normalize_text(text)
    if not text:
        return 0.0

    tokens = text.split()
    rep_tok = 0.0
    if len(tokens) > 1:
        rep_tok = sum(tokens[i] == tokens[i - 1] for i in range(1, len(tokens))) / max(1, len(tokens) - 1)

    max_run = 1
    cur = 1
    for i in range(1, len(text)):
        if text[i] == text[i - 1]:
            cur += 1
            max_run = max(max_run, cur)
        else:
            cur = 1

    run_pen = 1.0 if max_run >= 5 else 0.0
    return float(min(1.0, rep_tok + run_pen))


def compute_text_features(text, bbox=None, H=None, W=None, rec_conf=None, det_score=None):
    text = postprocess_ocr_text(text)
    rec_conf_norm = normalize_conf_value(rec_conf)
    if rec_conf_norm is None:
        rec_conf_norm = CONFIG.get('rec_conf_fallback_when_missing', 0.50)

    det_norm = normalize_conf_value(det_score)
    if det_norm is None:
        det_norm = 0.50

    lex = lexical_features(text)
    ch_pen = charset_penalty(text)
    rep_pen = repeated_garbage_penalty(text)
    weak_detection = det_norm < CONFIG.get('weak_det_score_threshold', 0.60)

    weights = CONFIG.get('feature_weights', {})
    score = CONFIG.get('composite_bias', 0.0)
    score += weights.get('rec_conf', 0.0) * rec_conf_norm
    score += weights.get('det_score', 0.0) * det_norm
    score += weights.get('lex_ratio', 0.0) * lex['lex_ratio']
    score += weights.get('diacritic_susp', 0.0) * lex['diacritic_susp']
    score += weights.get('charset_penalty', 0.0) * ch_pen
    score -= 0.08 * rep_pen
    score = float(np.clip(score, 0.0, 1.0))

    content_value = min(
        CONFIG.get('content_value_max_bonus', 0.20),
        0.015 * len(lex['tokens']) + 0.001 * len(text)
    )
    priority = float((1.0 - score) + CONFIG.get('content_value_weight', 0.10) * content_value)

    return {
        'rec_conf': float(rec_conf_norm),
        'det_score_norm': float(det_norm),
        'lex_ratio': float(lex['lex_ratio']),
        'diacritic_susp': float(lex['diacritic_susp']),
        'charset_penalty': float(ch_pen),
        'repetition_penalty': float(rep_pen),
        'composite_score': score,
        'quality_score': float(score * 100.0),
        'weak_detection': bool(weak_detection),
        'priority': priority,
        'content_value': float(content_value),
        'tokens': lex['tokens'],
        'eval_tokens': lex['eval_tokens'],
        'diacritic_suspicious_tokens': lex['diacritic_suspicious_tokens'],
        'oov_tokens': lex['oov_tokens'],
        'wordlist_enabled': bool(WORDLIST_ENABLED),
    }


def text_quality_score(text, bbox=None, H=None, W=None, rec_conf=None, det_score=None):
    return compute_text_features(text, bbox, H, W, rec_conf=rec_conf, det_score=det_score)['quality_score']


def is_structural_filtered(text, bbox, H, W):
    text = postprocess_ocr_text(text)
    if not text:
        return 'empty'
    if is_timestamp_text(text):
        return 'timestamp'
    if is_top_right_logo_region(bbox, H, W) and len(text) <= 10:
        return 'logo'
    if is_bottom_counter(text, bbox, H, W):
        return 'bottom_counter'
    return None


def classify_line_filter_status(line, H, W):
    text = postprocess_ocr_text(line.get('vietocr_text', ''))
    bbox = line.get('bbox_xyxy')

    structural = is_structural_filtered(text, bbox, H, W)
    if structural:
        return structural

    f = line.get('features') or compute_text_features(
        text, bbox, H, W,
        rec_conf=line.get('rec_conf'),
        det_score=line.get('det_score'),
    )
    line['features'] = f

    if (
        f['rec_conf'] >= CONFIG.get('auto_accept_rec_conf', 0.90)
        and f['det_score_norm'] >= CONFIG.get('auto_accept_det_score', 0.70)
        and f['lex_ratio'] >= CONFIG.get('auto_accept_lex_ratio', 0.50)
        and f['diacritic_susp'] <= CONFIG.get('auto_accept_max_diacritic_susp', 0.0)
        and f['charset_penalty'] <= CONFIG.get('auto_accept_max_charset_penalty', 0.05)
    ):
        return 'auto_accept'

    if f['weak_detection']:
        return 'vlm_candidate'
    if f['composite_score'] < CONFIG.get('escalate_composite_threshold', 0.68):
        return 'vlm_candidate'
    if f['rec_conf'] < CONFIG.get('escalate_rec_conf_threshold', 0.85):
        return 'vlm_candidate'
    if f['lex_ratio'] < CONFIG.get('escalate_lex_ratio_threshold', 0.45):
        return 'vlm_candidate'
    if f['diacritic_susp'] > CONFIG.get('escalate_diacritic_susp_threshold', 0.30):
        return 'vlm_candidate'

    return 'auto_accept'


def should_send_to_vintern(line):
    return line.get('filter_status') == 'vlm_candidate'


def text_similarity(a, b):
    a = normalize_text(a).lower()
    b = normalize_text(b).lower()
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return float(difflib.SequenceMatcher(None, a, b).ratio())


def choose_between_vietocr_and_vintern(line, vtext, vfeatures):
    old_text = normalize_text(line.get('vietocr_text', ''))
    old_score = float(line.get('composite_score', 0.0))
    vtext = postprocess_ocr_text(vtext)
    vscore = float(vfeatures.get('composite_score', 0.0))

    sim = text_similarity(old_text, vtext)
    agree = sim >= CONFIG.get('vintern_agreement_similarity', 0.82)

    if agree:
        if vscore >= old_score:
            return vtext, 'vintern_agree', True, False, sim
        return old_text, 'vietocr_agree', True, False, sim

    margin = CONFIG.get('vintern_score_margin_to_override', 0.06)
    if (
        vscore >= CONFIG.get('vintern_min_composite_accept', 0.68)
        and vscore >= old_score + margin
    ):
        return vtext, 'vintern_disagree_better_score', True, CONFIG.get('vintern_keep_review_on_disagreement', True), sim

    return old_text, 'vietocr_disagree_kept', False, True, sim


def vintern_prompt():
    return (
        "<image>\n"
        "Hãy đọc chính xác toàn bộ chữ trong ảnh crop này. "
        "Chỉ trả về nội dung OCR, không giải thích. "
        "Giữ nguyên tiếng Việt có dấu nếu có."
    )


def vintern_ocr_crop(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = dynamic_preprocess(
        image,
        image_size=CONFIG['vintern_image_size'],
        max_num=CONFIG['vintern_max_tiles'],
    ).to(VINTERN_DTYPE).to(VINTERN_DEVICE)
    generation_config = dict(max_new_tokens=CONFIG['vintern_max_new_tokens'], do_sample=False, num_beams=1)
    with torch.no_grad():
        response = vintern_model.chat(vintern_tokenizer, pixel_values, vintern_prompt(), generation_config)
    return normalize_text(response)


In [ ]:
# CELL 8.2 — V2 overrides: rec-missing gating, group-level Vintern, safe group text
GIAY_VARIANTS = {
    'giây', 'giay', 'giấy', 'gầy', '(gầy', 'gày', 'giy',
    'gẫy', 'gạy', 'gay', 'gầy)', '(giây', '(giay'
}


def is_60giay_noise(text, bbox, H, W):
    if not CONFIG.get('filter_60giay_variants', True):
        return False
    text_norm = normalize_text(text).lower().strip(' .,:;!?-_/()[]{}')
    if not isinstance(bbox, list) or len(bbox) != 4:
        return False
    x1, y1, x2, y2 = bbox
    if not (y1 >= H * 0.70 and len(text_norm) <= 6):
        return False
    return text_norm in GIAY_VARIANTS or text_norm in {'60', '6o', 'go', 'g0', '69'}


def is_structural_filtered(text, bbox, H, W):
    text = postprocess_ocr_text(text)
    if not text:
        return 'empty'
    if is_timestamp_text(text):
        return 'timestamp'
    if is_60giay_noise(text, bbox, H, W):
        return 'bottom_counter'
    if is_top_right_logo_region(bbox, H, W) and len(text) <= 10:
        return 'logo'
    if is_bottom_counter(text, bbox, H, W):
        return 'bottom_counter'
    return None


def rec_conf_is_unavailable(line):
    src = str(line.get('rec_conf_source', ''))
    return src in {
        'return_prob_missing', 'return_prob_no_conf', 'return_prob_list_missing',
        'text_only', 'error', 'no_crop'
    } or bool(line.get('rec_conf_flat_run', False))


def classify_line_filter_status(line, H, W):
    text = postprocess_ocr_text(line.get('vietocr_text', ''))
    bbox = line.get('bbox_xyxy')
    structural = is_structural_filtered(text, bbox, H, W)
    if structural:
        return structural
    f = line.get('features') or compute_text_features(
        text, bbox, H, W, rec_conf=line.get('rec_conf'), det_score=line.get('det_score')
    )
    line['features'] = f

    if rec_conf_is_unavailable(line):
        token_count = len(f.get('eval_tokens', []))
        can_auto_accept_long_text = (
            token_count >= CONFIG['rec_missing_auto_accept_min_tokens']
            and f['det_score_norm'] >= CONFIG['rec_missing_auto_accept_det_score']
            and f['lex_ratio'] >= CONFIG['rec_missing_auto_accept_lex_ratio']
            and f['diacritic_susp'] <= CONFIG['rec_missing_auto_accept_max_diacritic_susp']
            and f['charset_penalty'] <= CONFIG['rec_missing_auto_accept_max_charset_penalty']
            and not f['weak_detection']
        )
        if can_auto_accept_long_text:
            return 'auto_accept_rec_missing_long_lex'
        return 'vlm_candidate'

    if (
        f['rec_conf'] >= CONFIG.get('auto_accept_rec_conf', 0.90)
        and f['det_score_norm'] >= CONFIG.get('auto_accept_det_score', 0.70)
        and f['lex_ratio'] >= CONFIG.get('auto_accept_lex_ratio', 0.50)
        and f['diacritic_susp'] <= CONFIG.get('auto_accept_max_diacritic_susp', 0.0)
        and f['charset_penalty'] <= CONFIG.get('auto_accept_max_charset_penalty', 0.05)
    ):
        return 'auto_accept'
    if f['weak_detection']:
        return 'vlm_candidate'
    if f['composite_score'] < CONFIG.get('escalate_composite_threshold', 0.68):
        return 'vlm_candidate'
    if f['rec_conf'] < CONFIG.get('escalate_rec_conf_threshold', 0.85):
        return 'vlm_candidate'
    if f['lex_ratio'] < CONFIG.get('escalate_lex_ratio_threshold', 0.45):
        return 'vlm_candidate'
    if f['diacritic_susp'] > CONFIG.get('escalate_diacritic_susp_threshold', 0.30):
        return 'vlm_candidate'
    return 'auto_accept'


def build_group_text_keep_only(group, line_items, text_key='final_text'):
    lines = [line_items[i] for i in group['line_indices']]
    lines = sorted(lines, key=lambda x: ((x['bbox_xyxy'][1] + x['bbox_xyxy'][3]) / 2, x['bbox_xyxy'][0]))
    texts = []
    for line in lines:
        if not line.get('keep_for_index', False):
            continue
        txt = normalize_text(line.get(text_key, ''))
        if txt:
            texts.append(txt)
    return '\n'.join(texts)


def build_group_text_review_only(group, line_items, text_key='final_text'):
    lines = [line_items[i] for i in group['line_indices']]
    lines = sorted(lines, key=lambda x: ((x['bbox_xyxy'][1] + x['bbox_xyxy'][3]) / 2, x['bbox_xyxy'][0]))
    texts = []
    for line in lines:
        txt = normalize_text(line.get(text_key, ''))
        if txt:
            texts.append(txt)
    return '\n'.join(texts)


def vintern_ocr_group_crop(image_path):
    image = Image.open(image_path).convert('RGB')
    pixel_values = dynamic_preprocess(
        image,
        image_size=CONFIG['vintern_image_size'],
        max_num=CONFIG['vintern_max_tiles'],
    ).to(VINTERN_DTYPE).to(VINTERN_DEVICE)
    prompt = (
        '<image>\n'
        'Hãy đọc chính xác toàn bộ chữ trong ảnh crop này theo đúng từng dòng. '
        'Nếu ảnh có nhiều dòng chữ, hãy xuống dòng giữa các dòng. '
        'Chỉ trả về nội dung OCR, không giải thích.'
    )
    generation_config = dict(
        max_new_tokens=CONFIG.get('vintern_group_max_new_tokens', 160),
        do_sample=False,
        num_beams=1,
    )
    with torch.no_grad():
        response = vintern_model.chat(vintern_tokenizer, pixel_values, prompt, generation_config)
    return normalize_text(str(response).replace('\\n', '\n'))


def group_vlm_priority(group, line_items):
    refs = [line_items[i] for i in group['line_indices']]
    if not refs:
        return 0.0
    mean_priority = float(np.mean([float(x.get('priority', 0.0)) for x in refs]))
    num_vlm = sum(1 for x in refs if x.get('send_to_vintern'))
    num_review = sum(1 for x in refs if x.get('need_review'))
    num_lines = len(refs)
    bonus = 0.0
    if num_lines >= CONFIG.get('vintern_group_min_lines', 2): bonus += 0.25
    if num_vlm >= CONFIG.get('group_vlm_min_vlm_lines', 1): bonus += 0.20
    if num_review > 0: bonus += 0.15
    return mean_priority + bonus


def should_send_group_to_vintern(group, line_items):
    if not CONFIG.get('use_vintern_group_fallback', True): return False
    if not group.get('group_crop_path'): return False
    if group.get('num_lines', 0) < CONFIG.get('vintern_group_min_lines', 2): return False
    refs = [line_items[i] for i in group['line_indices']]
    if CONFIG.get('group_vlm_need_review_only', True):
        if not any(x.get('need_review') or x.get('send_to_vintern') for x in refs):
            return False
    return True


def group_vintern_should_accept(g, gv_text, gv_features):
    gv_text = normalize_text(gv_text)
    if not gv_text or gv_text.startswith('[GROUP_VINTERN_ERROR]'):
        return False
    score = float(gv_features.get('composite_score', 0.0))
    lex_ratio = float(gv_features.get('lex_ratio', 0.0))
    charset_pen = float(gv_features.get('charset_penalty', 1.0))
    if score >= CONFIG.get('group_vintern_min_composite_accept', 0.55):
        return True
    review_text = normalize_text(g.get('group_text_review', ''))
    if (
        CONFIG.get('group_vintern_accept_even_if_disagree', True)
        and len(gv_text) >= max(6, len(review_text) * 0.6)
        and lex_ratio >= 0.45
        and charset_pen <= 0.05
    ):
        return True
    return False

print('V2 override functions loaded.')


In [ ]:
# CELL 9 — Run detection
start = time.time()
det_output = detector.predict(IMAGE_PATH, batch_size=1)
det_time = time.time() - start
boxes, det_scores = parse_text_detection_output(det_output)
print(f'Detection done in {det_time:.3f}s')
print('Detected line boxes:', len(boxes))
for res in det_output:
    res.print()


In [ ]:
# CELL 10 — Build line items, group stacked boxes, crop line and group context
line_crop_dir = Path('/content/ppocr_line_perspective_crops')
group_crop_dir = Path('/content/ppocr_stacked_group_crops')
line_crop_dir.mkdir(parents=True, exist_ok=True)
group_crop_dir.mkdir(parents=True, exist_ok=True)
line_items = []

for i, (box, score) in enumerate(zip(boxes, det_scores)):
    if score is not None and score < CONFIG['drop_low_det_score_below']: continue
    box = np.array(box, dtype=np.float32)
    bw, bh = box_hw(box); aspect = bw/max(bh,1)
    if bw < CONFIG['min_box_width'] or bh < CONFIG['min_box_height'] or aspect < CONFIG['min_box_aspect_ratio']: continue
    item = {'line_id': len(line_items), 'det_idx': i, 'box': box.tolist(), 'bbox_xyxy': poly_to_xyxy(box), 'det_score': score, 'box_width': float(bw), 'box_height': float(bh), 'box_aspect': float(aspect)}
    crop = crop_perspective_line(img_rgb, box, CONFIG['perspective_padding'])
    pil = prepare_crop_for_ocr(crop, CONFIG['crop_min_height_for_ocr'], CONFIG['crop_upscale_max_factor'], CONFIG['add_white_border'], CONFIG['contrast_factor'])
    if pil is not None:
        path = line_crop_dir / f"line_{len(line_items):03d}_persp.png"
        pil.save(path)
        item['persp_crop_path'] = str(path); item['persp_crop_width'] = pil.size[0]; item['persp_crop_height'] = pil.size[1]
    line_items.append(item)

print('Line items:', len(line_items))

groups = build_stacked_groups(line_items, CONFIG)
line_items = assign_group_info_to_lines(line_items, groups)
print('Groups:', len(groups))
print('Multiline groups:', sum(g['is_multiline_group'] for g in groups))

for g in groups:
    if not g.get('is_multiline_group', False): continue
    crop = crop_axis_from_bbox(img_rgb, g['bbox_xyxy'], CONFIG['group_crop_padding'])
    pil = prepare_crop_for_ocr(crop, CONFIG['crop_min_height_for_ocr'], CONFIG['crop_upscale_max_factor'], CONFIG['add_white_border'], CONFIG['contrast_factor'])
    if pil is not None:
        path = group_crop_dir / f"group_{g['group_id']:03d}.png"
        pil.save(path)
        g['group_crop_path'] = str(path); g['group_crop_width'] = pil.size[0]; g['group_crop_height'] = pil.size[1]
        for idx in g['line_indices']: line_items[idx]['group_crop_path'] = str(path)

display(pd.DataFrame(line_items).head())
display(pd.DataFrame(groups))


In [ ]:
# CELL 11 — Visualize boxes, line crops and group crops
vis_boxes = img_rgb.copy()
for line in line_items:
    pts = np.array(line['box'], dtype=np.int32).reshape(-1,2)
    cv2.polylines(vis_boxes, [pts], True, (0,255,0), 2)
for g in groups:
    x1,y1,x2,y2 = [int(v) for v in g['bbox_xyxy']]
    color = (255,0,0) if g.get('is_multiline_group') else (0,0,255)
    cv2.rectangle(vis_boxes, (x1,y1), (x2,y2), color, 2)
    cv2.putText(vis_boxes, f"G{g['group_id']}|L{g['num_lines']}", (x1, max(0,y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
plt.figure(figsize=(16,9)); plt.imshow(vis_boxes); plt.axis('off'); plt.title('Green=line boxes | Blue/red=groups'); plt.show()

for line in line_items[:CONFIG['max_visualize_line_crops']]:
    p = line.get('persp_crop_path')
    if isinstance(p, str) and Path(p).exists():
        crop = Image.open(p).convert('RGB')
        plt.figure(figsize=(10,2.5)); plt.imshow(crop); plt.axis('off'); plt.title(f"line_id={line['line_id']} | group={line.get('group_id')} | det={line.get('det_score'):.3f}"); plt.show()

for g in [x for x in groups if isinstance(x.get('group_crop_path'), str) and Path(x['group_crop_path']).exists()][:CONFIG['max_visualize_groups']]:
    crop = Image.open(g['group_crop_path']).convert('RGB')
    plt.figure(figsize=(10,3.5)); plt.imshow(crop); plt.axis('off'); plt.title(f"group_id={g['group_id']} | lines={g['num_lines']}"); plt.show()


In [ ]:
# CELL 12 — VietOCR confidence + composite gating + ranked line Vintern
rec_times, vintern_times = [], []
rec_confs = []


def parse_vietocr_return_prob_output(out):
    if isinstance(out, tuple) and len(out) >= 2:
        text, prob = out[0], out[1]
        conf = normalize_conf_value(prob)
        return text, conf, 'return_prob_valid' if conf is not None else 'return_prob_missing'
    if isinstance(out, list) and len(out) >= 2:
        text, prob = out[0], out[1]
        conf = normalize_conf_value(prob)
        return text, conf, 'return_prob_list_valid' if conf is not None else 'return_prob_list_missing'
    return out, None, 'return_prob_no_conf'


def vietocr_predict_with_conf(crop):
    if CONFIG.get('use_vietocr_return_prob', True):
        try:
            out = vietocr_predictor.predict(crop, return_prob=True)
            text, conf, source = parse_vietocr_return_prob_output(out)
            return text, conf, source
        except TypeError:
            pass
        except Exception as e:
            print('WARNING: VietOCR return_prob failed:', repr(e))
    text = vietocr_predictor.predict(crop)
    return text, None, 'text_only'


def apply_features_and_status(line, text, rec_conf_norm):
    features = compute_text_features(text, line.get('bbox_xyxy'), H, W, rec_conf=rec_conf_norm, det_score=line.get('det_score'))
    line['features'] = features
    for k in [
        'lex_ratio', 'diacritic_susp', 'charset_penalty', 'repetition_penalty',
        'composite_score', 'quality_score', 'weak_detection', 'priority',
        'content_value', 'det_score_norm', 'tokens', 'eval_tokens',
        'diacritic_suspicious_tokens', 'oov_tokens', 'wordlist_enabled'
    ]:
        line[k] = features.get(k)
    status = classify_line_filter_status(line, H, W)
    line['filter_status'] = status
    line['is_filtered'] = status in {'timestamp', 'logo', 'bottom_counter', 'short_noise', 'empty', 'no_crop'}
    line['send_to_vintern'] = should_send_to_vintern(line)
    if line['is_filtered']:
        line.update({'final_text': '', 'final_source': f'filtered_{status}', 'keep_for_index': False, 'need_review': False})
    elif status.startswith('auto_accept'):
        line.update({'final_text': text, 'final_source': status, 'keep_for_index': True, 'need_review': False})
    else:
        line.update({'final_text': text, 'final_source': 'vietocr_vlm_candidate', 'keep_for_index': False, 'need_review': True})


for line in tqdm(line_items, desc='VietOCR line crops + composite gating'):
    p = line.get('persp_crop_path')
    if not isinstance(p, str) or not Path(p).exists():
        line.update({
            'vietocr_text': '', 'rec_conf': None, 'rec_conf_source': 'no_crop', 'vietocr_time_sec': None,
            'features': {}, 'composite_score': 0.0, 'quality_score': -999, 'lex_ratio': 0.0,
            'diacritic_susp': 0.0, 'charset_penalty': 1.0, 'weak_detection': True, 'priority': 1.0,
            'filter_status': 'no_crop', 'is_filtered': True, 'send_to_vintern': False,
            'final_text': '', 'final_source': 'no_crop', 'keep_for_index': False, 'need_review': False,
        })
        continue
    crop = Image.open(p).convert('RGB')
    start = time.time()
    try:
        text, rec_conf, rec_conf_source = vietocr_predict_with_conf(crop)
    except Exception as e:
        text = f'[VIETOCR_ERROR] {repr(e)}'
        rec_conf = None
        rec_conf_source = 'error'
    elapsed = time.time() - start
    text = postprocess_ocr_text(text)
    rec_conf_norm = normalize_conf_value(rec_conf)
    if rec_conf_norm is not None:
        rec_confs.append(rec_conf_norm)
    line['vietocr_text'] = text
    line['rec_conf'] = rec_conf_norm if rec_conf_norm is not None else CONFIG.get('rec_conf_fallback_when_missing', 0.50)
    line['rec_conf_source'] = rec_conf_source
    line['vietocr_time_sec'] = elapsed
    line['rec_conf_flat_run'] = False
    apply_features_and_status(line, text, rec_conf_norm)
    rec_times.append(elapsed)

# Detect missing/flat confidence
rec_conf_flat = False
if rec_confs:
    s = pd.Series(rec_confs)
    print('rec_conf stats:')
    print(s.describe())
    rec_std = float(s.std()) if len(s) > 1 else 0.0
    unique_ratio = float(s.round(6).nunique() / max(1, len(s)))
    rec_conf_flat = (
        rec_std <= CONFIG.get('rec_conf_flat_std_threshold', 1e-4)
        or unique_ratio <= CONFIG.get('rec_conf_flat_unique_ratio_threshold', 0.20)
    )
    print({'rec_conf_std': rec_std, 'rec_conf_unique_ratio': unique_ratio, 'rec_conf_flat_detected': rec_conf_flat})
    plt.figure(figsize=(8, 4))
    plt.hist(rec_confs, bins=20)
    plt.title('VietOCR rec_conf distribution')
    plt.xlabel('rec_conf')
    plt.ylabel('count')
    plt.show()
else:
    print('No rec_conf collected. VietOCR return_prob may not be supported.')
    rec_conf_flat = True

if rec_conf_flat:
    print('WARNING: rec_conf appears flat/unreliable. Recomputing status with rec_conf_flat_run=True.')
    for line in line_items:
        if line.get('is_filtered'):
            continue
        line['rec_conf_flat_run'] = True
        apply_features_and_status(line, line.get('vietocr_text', ''), line.get('rec_conf', None))

vlm_candidates = [x for x in line_items if x.get('send_to_vintern')]
candidates = sorted(vlm_candidates, key=lambda x: -float(x.get('priority', 0.0)))[:CONFIG['vintern_max_candidates']]
print('Vintern line candidates total:', len(vlm_candidates))
print('Vintern line candidates selected:', len(candidates))
if candidates:
    display(pd.DataFrame(candidates)[[
        'line_id', 'vietocr_text', 'composite_score', 'priority',
        'rec_conf', 'rec_conf_source', 'det_score', 'lex_ratio',
        'diacritic_susp', 'charset_penalty', 'weak_detection', 'persp_crop_path'
    ]])

if CONFIG['use_vintern_fallback'] and USE_VINTERN and vintern_model is not None:
    for line in tqdm(candidates, desc='Vintern line fallback ranked'):
        start = time.time()
        try:
            vtext = vintern_ocr_crop(line['persp_crop_path'])
        except Exception as e:
            vtext = f'[VINTERN_ERROR] {repr(e)}'
        elapsed = time.time() - start
        vtext = postprocess_ocr_text(vtext)
        vfeatures = compute_text_features(vtext, line.get('bbox_xyxy'), H, W, rec_conf=0.70, det_score=line.get('det_score'))
        chosen_text, chosen_source, keep, need_review, agreement_sim = choose_between_vietocr_and_vintern(line, vtext, vfeatures)
        line['vintern_text'] = vtext
        line['vintern_features'] = vfeatures
        line['vintern_composite_score'] = vfeatures.get('composite_score')
        line['vintern_quality_score'] = vfeatures.get('quality_score')
        line['vintern_lex_ratio'] = vfeatures.get('lex_ratio')
        line['vintern_diacritic_susp'] = vfeatures.get('diacritic_susp')
        line['vintern_time_sec'] = elapsed
        line['agreement_similarity'] = agreement_sim
        line.update({'final_text': chosen_text, 'final_source': chosen_source, 'keep_for_index': bool(keep), 'need_review': bool(need_review)})
        vintern_times.append(elapsed)
else:
    print('Vintern line fallback disabled or model not loaded.')

rec_total_time = float(sum(rec_times))
rec_avg_time = float(np.mean(rec_times)) if rec_times else 0.0
vintern_total_time = float(sum(vintern_times))
vintern_avg_time = float(np.mean(vintern_times)) if vintern_times else 0.0
print(f'VietOCR total time: {rec_total_time:.3f}s | avg: {rec_avg_time:.3f}s')
print(f'Vintern line total time: {vintern_total_time:.3f}s | avg: {vintern_avg_time:.3f}s')
num_lines = len(line_items)
num_structural = sum(1 for x in line_items if x.get('is_filtered'))
num_escalate = len(vlm_candidates)
num_auto = sum(1 for x in line_items if str(x.get('filter_status', '')).startswith('auto_accept'))
print({
    'num_lines': num_lines,
    'num_structural_filtered': num_structural,
    'num_auto_accept': num_auto,
    'num_escalate_candidates': num_escalate,
    'escalation_rate_non_structural': num_escalate / max(1, num_lines - num_structural),
    'rec_conf_flat': rec_conf_flat,
})


In [ ]:
# CELL 13 — Build group_text safely: clean vs review
for g in groups:
    review_text = build_group_text_review_only(g, line_items, text_key='final_text')
    clean_text = build_group_text_keep_only(g, line_items, text_key='final_text')
    refs = [line_items[i] for i in g['line_indices']]
    g['group_text_review'] = review_text
    g['group_text_clean'] = clean_text
    g['group_text'] = clean_text if (clean_text or CONFIG.get('allow_group_text_fallback_to_review', False)) else ''
    if CONFIG.get('allow_group_text_fallback_to_review', False) and not clean_text:
        g['group_text'] = review_text
    g['group_text_lines'] = g['group_text'].split('\n') if g['group_text'] else []
    g['group_review_lines'] = review_text.split('\n') if review_text else []
    g['num_keep_lines'] = sum(1 for x in refs if x.get('keep_for_index'))
    g['num_filtered_lines'] = sum(1 for x in refs if x.get('is_filtered'))
    g['num_vlm_candidates'] = sum(1 for x in refs if x.get('send_to_vintern'))
    g['num_vintern_lines'] = sum(1 for x in refs if str(x.get('final_source', '')).startswith('vintern'))
    g['need_review'] = any(x.get('need_review', False) for x in refs)
    g['mean_composite_score'] = float(np.mean([x.get('composite_score', 0.0) for x in refs])) if refs else 0.0
    g['mean_rec_conf'] = float(np.mean([x.get('rec_conf', 0.0) for x in refs])) if refs else 0.0
    g['group_vlm_priority'] = group_vlm_priority(g, line_items)

text_map = {g['group_id']: g.get('group_text', '') for g in groups}
review_map = {g['group_id']: g.get('group_text_review', '') for g in groups}
for line in line_items:
    line['group_text'] = text_map.get(line.get('group_id'), '')
    line['group_text_review'] = review_map.get(line.get('group_id'), '')

df_lines = pd.DataFrame(line_items)
df_groups = pd.DataFrame(groups)

display_cols_lines = [
    'line_id', 'group_id', 'group_num_lines', 'is_multiline_group',
    'vietocr_text', 'rec_conf', 'rec_conf_source', 'rec_conf_flat_run', 'det_score',
    'composite_score', 'quality_score', 'lex_ratio', 'diacritic_susp',
    'diacritic_suspicious_tokens', 'oov_tokens',
    'charset_penalty', 'repetition_penalty', 'weak_detection', 'priority',
    'filter_status', 'send_to_vintern',
    'vintern_text', 'vintern_composite_score', 'vintern_quality_score',
    'agreement_similarity', 'final_text', 'final_source', 'keep_for_index', 'need_review',
    'group_text', 'group_text_review', 'persp_crop_path', 'group_crop_path', 'bbox_xyxy'
]

display_cols_groups = [
    'reading_order', 'group_id', 'num_lines', 'is_multiline_group',
    'group_text', 'group_text_clean', 'group_text_review',
    'group_text_lines', 'group_review_lines', 'group_crop_path',
    'num_keep_lines', 'num_filtered_lines', 'num_vlm_candidates', 'num_vintern_lines',
    'need_review', 'mean_composite_score', 'mean_rec_conf', 'group_vlm_priority',
    'bbox_xyxy', 'mean_det_score'
]

display(df_lines[[c for c in display_cols_lines if c in df_lines.columns]])
display(df_groups[[c for c in display_cols_groups if c in df_groups.columns]])


In [ ]:
# CELL 13.1 — Group-level Vintern fallback for multi-line context
group_vintern_times = []

group_candidates_all = [g for g in groups if should_send_group_to_vintern(g, line_items)]
group_candidates = sorted(group_candidates_all, key=lambda g: -float(g.get('group_vlm_priority', 0.0)))[:CONFIG.get('vintern_group_max_candidates', 4)]
print('Group Vintern candidates total:', len(group_candidates_all))
print('Group Vintern selected:', len(group_candidates))
if group_candidates:
    display(pd.DataFrame(group_candidates)[[
        'group_id', 'reading_order', 'num_lines', 'group_text_review', 'group_text_clean',
        'num_keep_lines', 'num_vlm_candidates', 'need_review', 'group_vlm_priority', 'group_crop_path'
    ]])

if CONFIG.get('use_vintern_group_fallback', True) and USE_VINTERN and vintern_model is not None:
    for g in tqdm(group_candidates, desc='Vintern group fallback'):
        start = time.time()
        try:
            gv_text = vintern_ocr_group_crop(g['group_crop_path'])
        except Exception as e:
            gv_text = f'[GROUP_VINTERN_ERROR] {repr(e)}'
        elapsed = time.time() - start
        gv_text = postprocess_ocr_text(gv_text)
        gv_features = compute_text_features(
            gv_text, g.get('bbox_xyxy'), H, W, rec_conf=0.72, det_score=g.get('mean_det_score', 0.70)
        )
        g['group_vintern_text'] = gv_text
        g['group_vintern_features'] = gv_features
        g['group_vintern_composite_score'] = gv_features.get('composite_score')
        g['group_vintern_quality_score'] = gv_features.get('quality_score')
        g['group_vintern_lex_ratio'] = gv_features.get('lex_ratio')
        g['group_vintern_diacritic_susp'] = gv_features.get('diacritic_susp')
        g['group_vintern_time_sec'] = elapsed
        old_text = normalize_text(g.get('group_text_clean', ''))
        review_text = normalize_text(g.get('group_text_review', ''))
        old_for_compare = old_text if old_text else review_text
        sim = text_similarity(old_for_compare, gv_text)
        g['group_vintern_agreement_similarity'] = sim
        accept_group_vintern = group_vintern_should_accept(g, gv_text, gv_features)
        if accept_group_vintern:
            g['group_text_clean'] = gv_text
            g['group_text'] = gv_text
            g['group_text_lines'] = gv_text.split('\n') if gv_text else []
            g['group_final_source'] = 'vintern_group_fallback'
            g['need_review'] = bool(sim < CONFIG.get('vintern_agreement_similarity', 0.82))
            g['group_keep_for_index'] = True
        else:
            g['group_final_source'] = 'line_level_or_review_only'
            g['group_keep_for_index'] = bool(g.get('group_text_clean', ''))
            g['need_review'] = True
        group_vintern_times.append(elapsed)
else:
    print('Group Vintern fallback disabled or model not loaded.')

group_vintern_total_time = float(sum(group_vintern_times))
group_vintern_avg_time = float(np.mean(group_vintern_times)) if group_vintern_times else 0.0
print(f'Vintern group total time: {group_vintern_total_time:.3f}s | avg: {group_vintern_avg_time:.3f}s')

text_map = {g['group_id']: g.get('group_text', '') for g in groups}
for line in line_items:
    line['group_text'] = text_map.get(line.get('group_id'), '')

df_groups = pd.DataFrame(groups)
display_cols_groups_after_vlm = [
    'reading_order', 'group_id', 'num_lines', 'group_text', 'group_text_review',
    'group_vintern_text', 'group_vintern_composite_score', 'group_vintern_agreement_similarity',
    'group_final_source', 'group_keep_for_index', 'need_review', 'group_crop_path'
]
display(df_groups[[c for c in display_cols_groups_after_vlm if c in df_groups.columns]])


In [ ]:
# CELL 14 — Visualize final OCR and group text
vis_final = img_rgb.copy()
for g in groups:
    x1,y1,x2,y2 = [int(v) for v in g['bbox_xyxy']]
    color = (255,0,0) if g.get('is_multiline_group') else (0,0,255)
    cv2.rectangle(vis_final, (x1,y1), (x2,y2), color, 2)
    cv2.putText(vis_final, f"G{g['group_id']}|L{g['num_lines']}", (x1, max(0,y1-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2, cv2.LINE_AA)
for line in line_items:
    box = np.array(line['box'], dtype=np.int32).reshape(-1,2)
    color = (150,150,150) if line.get('is_filtered') else ((0,255,0) if line.get('keep_for_index') else (255,165,0))
    cv2.polylines(vis_final, [box], True, color, 2)
    text = line.get('final_text','') or line.get('vietocr_text','')
    x,y = box[0]
    cv2.putText(vis_final, f"{line['line_id']}: {str(text)[:35]}", (int(x), max(0, int(y)-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1, cv2.LINE_AA)
plt.figure(figsize=(16,10)); plt.imshow(vis_final); plt.axis('off'); plt.title('Final OCR: green=keep, orange=review, gray=filtered'); plt.show()

for g in [x for x in groups if isinstance(x.get('group_crop_path'), str) and Path(x['group_crop_path']).exists()][:CONFIG['max_visualize_groups']]:
    crop = Image.open(g['group_crop_path']).convert('RGB')
    plt.figure(figsize=(10,3.5)); plt.imshow(crop); plt.axis('off'); plt.title(f"group_id={g['group_id']} | lines={g['num_lines']}\n{g.get('group_text','')}"); plt.show()


In [ ]:
# CELL 15 — Final clean/review text after group-level fallback
groups_sorted = sorted(groups, key=lambda g: g.get('reading_order', g['group_id']))
clean_group_texts = []
review_group_texts = []
for g in groups_sorted:
    clean = normalize_text(g.get('group_text_clean', ''))
    review = normalize_text(g.get('group_text_review', ''))
    if clean:
        clean_group_texts.append(clean)
    elif review:
        review_group_texts.append(review)
final_clean_text = '\n\n'.join(clean_group_texts)
final_review_text = '\n\n'.join(review_group_texts)
final_search_text = final_clean_text
print('===== FINAL CLEAN TEXT FOR INDEX =====')
print(final_clean_text)
print('\n===== REVIEW TEXT ONLY, NOT FOR CLEAN INDEX =====')
print(final_review_text)


In [ ]:
# CELL 16 — ES-ready document
frame_id = Path(IMAGE_PATH).stem
groups_sorted = sorted(groups, key=lambda g: g.get('reading_order', g['group_id']))

es_doc = {
    'frame_id': frame_id,
    'image_path': IMAGE_PATH,
    'ocr_pipeline': {
        'detector': CONFIG['det_model_name'],
        'line_recognizer': f"VietOCR/{CONFIG['vietocr_config']}",
        'fallback_vlm_line': CONFIG['vintern_model_name'] if CONFIG.get('use_vintern_fallback') else None,
        'fallback_vlm_group': CONFIG['vintern_model_name'] if CONFIG.get('use_vintern_group_fallback') else None,
        'wordlist_enabled': WORDLIST_ENABLED,
        'wordlist_size': len(VN_WORDSET),
        'group_text_rule': 'group_text_clean only; group_text_review is not indexed',
        'gating_config': {
            'feature_weights': CONFIG['feature_weights'],
            'rec_missing_auto_accept_min_tokens': CONFIG['rec_missing_auto_accept_min_tokens'],
            'rec_missing_auto_accept_det_score': CONFIG['rec_missing_auto_accept_det_score'],
            'rec_missing_auto_accept_lex_ratio': CONFIG['rec_missing_auto_accept_lex_ratio'],
            'group_vintern_min_composite_accept': CONFIG['group_vintern_min_composite_accept'],
        },
    },
    'timing': {
        'det_time_sec': det_time,
        'vietocr_total_time_sec': rec_total_time,
        'vietocr_avg_time_sec': rec_avg_time,
        'vintern_line_total_time_sec': vintern_total_time,
        'vintern_line_avg_time_sec': vintern_avg_time,
        'vintern_group_total_time_sec': group_vintern_total_time if 'group_vintern_total_time' in globals() else 0.0,
        'vintern_group_avg_time_sec': group_vintern_avg_time if 'group_vintern_avg_time' in globals() else 0.0,
    },
    'ocr_text_clean': final_search_text,
    'ocr_text_review': final_review_text,
    'ocr_group_texts_clean': clean_group_texts,
    'ocr_group_texts_review': review_group_texts,
    'ocr_lines': [
        {
            'line_id': line.get('line_id'),
            'group_id': line.get('group_id'),
            'vietocr_text': line.get('vietocr_text'),
            'rec_conf': line.get('rec_conf'),
            'rec_conf_source': line.get('rec_conf_source'),
            'rec_conf_flat_run': line.get('rec_conf_flat_run'),
            'vintern_text': line.get('vintern_text'),
            'final_text': line.get('final_text'),
            'final_source': line.get('final_source'),
            'composite_score': line.get('composite_score'),
            'lex_ratio': line.get('lex_ratio'),
            'diacritic_susp': line.get('diacritic_susp'),
            'oov_tokens': line.get('oov_tokens'),
            'filter_status': line.get('filter_status'),
            'is_filtered': line.get('is_filtered'),
            'send_to_vintern': line.get('send_to_vintern'),
            'keep_for_index': line.get('keep_for_index'),
            'need_review': line.get('need_review'),
            'det_score': line.get('det_score'),
            'box': line.get('box'),
            'bbox_xyxy': line.get('bbox_xyxy'),
            'persp_crop_path': line.get('persp_crop_path'),
            'group_crop_path': line.get('group_crop_path'),
        } for line in line_items
    ],
    'ocr_groups': [
        {
            'reading_order': g.get('reading_order'),
            'group_id': g.get('group_id'),
            'num_lines': g.get('num_lines'),
            'group_text': g.get('group_text'),
            'group_text_clean': g.get('group_text_clean'),
            'group_text_review': g.get('group_text_review'),
            'group_vintern_text': g.get('group_vintern_text'),
            'group_vintern_composite_score': g.get('group_vintern_composite_score'),
            'group_vintern_agreement_similarity': g.get('group_vintern_agreement_similarity'),
            'group_final_source': g.get('group_final_source'),
            'group_keep_for_index': g.get('group_keep_for_index'),
            'group_crop_path': g.get('group_crop_path'),
            'bbox_xyxy': g.get('bbox_xyxy'),
            'mean_det_score': g.get('mean_det_score'),
            'mean_composite_score': g.get('mean_composite_score'),
            'num_keep_lines': g.get('num_keep_lines'),
            'num_vlm_candidates': g.get('num_vlm_candidates'),
            'need_review': g.get('need_review'),
        } for g in groups_sorted
    ],
}
es_doc


In [ ]:
# CELL 17 — Save outputs
out_dir = Path('/content/ppocr_group_vietocr_vintern_wordlist_gating_v2_output')
out_dir.mkdir(parents=True, exist_ok=True)

lines_csv_path = out_dir / f'{Path(IMAGE_PATH).stem}_ocr_lines_wordlist_gating_v2.csv'
groups_csv_path = out_dir / f'{Path(IMAGE_PATH).stem}_ocr_groups_wordlist_gating_v2.csv'
json_path = out_dir / f'{Path(IMAGE_PATH).stem}_ocr_es_doc_wordlist_gating_v2.json'
txt_path = out_dir / f'{Path(IMAGE_PATH).stem}_ocr_clean_text_wordlist_gating_v2.txt'
review_txt_path = out_dir / f'{Path(IMAGE_PATH).stem}_ocr_review_text_wordlist_gating_v2.txt'
vis_path = out_dir / f'{Path(IMAGE_PATH).stem}_ocr_vis_wordlist_gating_v2.png'

pd.DataFrame(line_items).to_csv(lines_csv_path, index=False, encoding='utf-8-sig')
pd.DataFrame(groups).to_csv(groups_csv_path, index=False, encoding='utf-8-sig')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(to_jsonable(es_doc), f, ensure_ascii=False, indent=2)
with open(txt_path, 'w', encoding='utf-8') as f:
    f.write(final_clean_text)
with open(review_txt_path, 'w', encoding='utf-8') as f:
    f.write(final_review_text)
Image.fromarray(vis_final).save(vis_path)

print('Saved lines CSV:', lines_csv_path)
print('Saved groups CSV:', groups_csv_path)
print('Saved JSON:', json_path)
print('Saved CLEAN TXT:', txt_path)
print('Saved REVIEW TXT:', review_txt_path)
print('Saved visualization:', vis_path)


In [ ]:
# CELL 18 — Tuning notes
print("""
Tuning nhanh:

1. Nếu còn lỗi dynamic_preprocess:
   - Chạy lại từ CELL 6 và CELL 6.1.
   - Vintern thật sẽ không chạy nhanh bất thường kiểu 0.003s/crop.

2. Nếu VietOCR return_prob missing:
   - Xem cột rec_conf_source.
   - Nếu toàn return_prob_missing/text_only, pipeline sẽ dùng rec-missing rule:
     chỉ auto_accept dòng dài có det_score + lex_ratio tốt.
   - Dòng ngắn/tên riêng/biển báo sẽ đi line/group Vintern.

3. Nếu ticker dài vẫn bị escalate quá nhiều:
   - Giảm CONFIG['rec_missing_auto_accept_min_tokens'] từ 5 xuống 4.
   - Giảm CONFIG['rec_missing_auto_accept_lex_ratio'] từ 0.70 xuống 0.60.
   - Spot-check trước khi dùng production.

4. Nếu tên riêng/biển báo bị accept sai:
   - Tăng CONFIG['rec_missing_auto_accept_min_tokens'] lên 6.
   - Giữ group Vintern cho group nhiều dòng.

5. Nếu Vintern group đọc đúng nhưng không accept:
   - Giảm CONFIG['group_vintern_min_composite_accept'] từ 0.55 xuống 0.50.
   - Hoặc giảm yêu cầu lex_ratio trong group_vintern_should_accept.

6. group_text vs group_text_review:
   - group_text/group_text_clean: dùng để index chính.
   - group_text_review: chỉ để debug, không nên index chính.
""")
